<a href="https://colab.research.google.com/github/r4tangUCSD/151B_SP26_Competition/blob/SFT-dylan/qwen3__4b_thinking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-4b-thinking-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.5.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Qwen-3` format for conversation style finetunes. We use the [Open Math Reasoning](https://huggingface.co/datasets/unsloth/OpenMathReasoning-mini) dataset which was used to win the [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard) (AI Mathematical Olympiad - Progress Prize 2) challenge! We sample 10% of verifiable reasoning traces that used DeepSeek R1, and which got > 95% accuracy. Qwen-3 renders multi turn conversations like below:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [5]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-thinking",
)

In [6]:
# more SFT data with quality filters
from datasets import load_dataset


dataset = load_dataset("AI-MO/NuminaMath-CoT", split="train")
dataset = dataset.filter(lambda x: x['source'] in [
    'amc_aime', 'aops_forum', 'math', 'olympiads'
])
dataset = dataset.select(range(min(100000, len(dataset))))

We now convert the reasoning dataset into conversational format:

In [7]:
def generate_conversation(examples):
    conversations = []
    for messages in examples["messages"]:
        conversations.append(messages)
    return {"conversations": conversations}

dataset = dataset.map(generate_conversation, batched=True)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [8]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Let's see how the chat template did!

In [9]:
dataset[100]['text']

"<|im_start|>user\nThe sequence \\( x_{n} \\) has its first two elements as \\( x_{1}=1001 \\) and \\( x_{2}=1003 \\). For \\( n \\geq 1 \\), the recurrence relation is given by:\n\\[ x_{n+2}=\\frac{x_{n+1}-2004}{x_{n}}. \\]\nWhat is the sum of the first 2004 terms of the sequence?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n1. We start with the given sequence \\( x_{n} \\) where the initial terms are:\n   \\[\n   x_{1}=1001, \\quad x_{2}=1003\n   \\]\n   and for \\( n \\geq 1 \\):\n   \\[\n   x_{n+2}=\\frac{x_{n+1}-2004}{x_{n}}\n   \\]\n\n2. Notice that \\( x_{1} + x_{2} = 1001 + 1003 = 2004 \\). This relationship can be used to rewrite the recursion:\n   \\[\n   x_{n+2}=\\frac{x_{n+1}-x_{1}-x_{2}}{x_{n}} \n   \\]\n\n3. Let's compute a few terms of the sequence to identify any patterns:\n   \\[\n   x_{1}=1001, \\quad x_{2}=1003\n   \\]\n   \\[\n   x_{3}=\\frac{x_{2}-2004}{x_{1}} = \\frac{1003-2004}{1001} = \\frac{-1001}{1001} = -1 \n   \\]\n   \\[\n   x_{4}=\\frac{x_{3}-2

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [10]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # 2 epochs so more training
        # max_steps = 60,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/100000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [11]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=16):   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/100000 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [12]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

"<|im_start|>user\nThe sequence \\( x_{n} \\) has its first two elements as \\( x_{1}=1001 \\) and \\( x_{2}=1003 \\). For \\( n \\geq 1 \\), the recurrence relation is given by:\n\\[ x_{n+2}=\\frac{x_{n+1}-2004}{x_{n}}. \\]\nWhat is the sum of the first 2004 terms of the sequence?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n1. We start with the given sequence \\( x_{n} \\) where the initial terms are:\n   \\[\n   x_{1}=1001, \\quad x_{2}=1003\n   \\]\n   and for \\( n \\geq 1 \\):\n   \\[\n   x_{n+2}=\\frac{x_{n+1}-2004}{x_{n}}\n   \\]\n\n2. Notice that \\( x_{1} + x_{2} = 1001 + 1003 = 2004 \\). This relationship can be used to rewrite the recursion:\n   \\[\n   x_{n+2}=\\frac{x_{n+1}-x_{1}-x_{2}}{x_{n}} \n   \\]\n\n3. Let's compute a few terms of the sequence to identify any patterns:\n   \\[\n   x_{1}=1001, \\quad x_{2}=1003\n   \\]\n   \\[\n   x_{3}=\\frac{x_{2}-2004}{x_{1}} = \\frac{1003-2004}{1001} = \\frac{-1001}{1001} = -1 \n   \\]\n   \\[\n   x_{4}=\\frac{x_{3}-2

Now let's print the masked out example - you should see only the answer is present:

In [13]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                                                           <think>\n\n</think>\n\n1. We start with the given sequence \\( x_{n} \\) where the initial terms are:\n   \\[\n   x_{1}=1001, \\quad x_{2}=1003\n   \\]\n   and for \\( n \\geq 1 \\):\n   \\[\n   x_{n+2}=\\frac{x_{n+1}-2004}{x_{n}}\n   \\]\n\n2. Notice that \\( x_{1} + x_{2} = 1001 + 1003 = 2004 \\). This relationship can be used to rewrite the recursion:\n   \\[\n   x_{n+2}=\\frac{x_{n+1}-x_{1}-x_{2}}{x_{n}} \n   \\]\n\n3. Let's compute a few terms of the sequence to identify any patterns:\n   \\[\n   x_{1}=1001, \\quad x_{2}=1003\n   \\]\n   \\[\n   x_{3}=\\frac{x_{2}-2004}{x_{1}} = \\frac{1003-2004}{1001} = \\frac{-1001}{1001} = -1 \n   \\]\n   \\[\n   x_{4}=\\frac{x_{3}-2004}{x_{2}} = \\frac{-1-2004}{1003} = \\frac{-2005}{1003} = -1\n   \\]\n   \\[\n   x_{5}=\\frac{x_{4}-2004}{x_{3}} = \\frac{-1-2004}{-1} = \\frac{-2005}{-1} = 2005\n   \\]\n   \\[\n   x_{6}=\\frac{x_{5}-2004}{

In [14]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.251 GB.
3.816 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [15]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 2 | Total steps = 12,500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 16,515,072 of 4,038,983,168 (0.41% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.815100
20,0.740400
30,0.688400
40,0.607300
50,0.558700
60,0.586200
70,0.578500
80,0.541000
90,0.514300
100,0.529700


In [16]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

40250.5691 seconds used for training.
670.84 minutes used for training.
Peak reserved memory = 23.258 GB.
Peak reserved memory for training = 19.442 GB.
Peak reserved memory % of max memory = 29.347 %.
Peak reserved memory for training % of max memory = 24.532 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for instruct inference are `temperature = 0.7, top_p = 0.8, top_k = 20`

For reasoning chat based inference, `temperature = 0.6, top_p = 0.95, top_k = 20`

In [17]:
messages = [
    {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = True, # Disable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 450, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<|im_start|>user
Solve (x + 2)^2 = 0.<|im_end|>
<|im_start|>assistant
<think>
</think>

To solve the equation $(x + 2)^2 = 0$, we follow these steps:

1. Recognize that for the square of a number to be zero, the number itself must be zero. This is because the square of any non-zero number is always positive, and the square of zero is zero. Therefore, we can write:
   \[
   (x + 2)^2 = 0 \implies x + 2 = 0
   \]

2. Solve for $x$ by subtracting 2 from both sides of the equation:
   \[
   x + 2 - 2 = 0 - 2 \implies x = -2
   \]

Therefore, the solution to the equation $(x + 2)^2 = 0$ is $\boxed{x = -2}$.<|im_end|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [18]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'UCSD-Seal-Logo.png', 'UCSD_Seal.png', 'Nutrition.gdoc', 'Cogs 109 Checkpoint 2 - Dylan Craver.gslides', 'checkpoint2 - dylan .mov', 'checkpoint3 - dylan.mp4', 'Cogs 109 Checkpoint 3 - Dylan Craver.gslides', 'Untitled document (1).gdoc', 'KNN.ipynb', 'Sickle Results Directions.docx', 'Dylan_Craver_Resume (15) (1).pdf', 'Dylan_Craver_Resume (15).pdf', 'DSC100 HW4.gdoc', 'Untitled document.gdoc', 'proj3', '151B_SP26_Competition-main.zip', '151B_SP26_Competition-main', 'responses_n10.json', 'r8_a16_numinamath_lora', 'r8_a16_numinamath_16bit']


In [19]:
from google.colab import userdata
token = userdata.get('HT_TOKEN')

In [20]:
# Save to Drive (persistent)
model.save_pretrained("/content/drive/MyDrive/r8_a16_numinamath_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/r8_a16_numinamath_lora")

# Push to HuggingFace (shareable with teammates)
model.push_to_hub("dcraver2005/r8_a16_numinamath_lora", token=token)
tokenizer.push_to_hub("dcraver2005/r8_a16_numinamath_lora", token=token)

README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 51.4kB / 66.1MB            

Saved model to https://huggingface.co/dcraver2005/r8_a16_numinamath_lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpbbs175x_/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [21]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 16384,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [22]:
# Save merged 16bit to Drive
if True:
    model.save_pretrained_merged("/content/drive/MyDrive/r8_a16_numinamath_16bit", tokenizer, save_method="merged_16bit")

if True:
    from google.colab import userdata
    token = userdata.get('HT_TOKEN')
    model.push_to_hub_merged("dcraver2005/r8_a16_numinamath_16bit", tokenizer, save_method="merged_16bit", token=token)

config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:19<00:00,  9.83s/it]


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          | 4.22MB / 3.08GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:24<00:00, 72.48s/it]


Unsloth: Merge process complete. Saved to `/content/dcraver2005/r8_a16_numinamath_16bit`


In [23]:
!df -h /content/drive/MyDrive

Filesystem      Size  Used Avail Use% Mounted on
drive           236G   75G  162G  32% /content/drive


In [24]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'UCSD-Seal-Logo.png', 'UCSD_Seal.png', 'Nutrition.gdoc', 'Cogs 109 Checkpoint 2 - Dylan Craver.gslides', 'checkpoint2 - dylan .mov', 'checkpoint3 - dylan.mp4', 'Cogs 109 Checkpoint 3 - Dylan Craver.gslides', 'Untitled document (1).gdoc', 'KNN.ipynb', 'Sickle Results Directions.docx', 'Dylan_Craver_Resume (15) (1).pdf', 'Dylan_Craver_Resume (15).pdf', 'DSC100 HW4.gdoc', 'Untitled document.gdoc', 'proj3', '151B_SP26_Competition-main.zip', '151B_SP26_Competition-main', 'responses_n10.json', 'r8_a16_numinamath_lora', 'r8_a16_numinamath_16bit']


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [25]:
# # Save to 8bit Q8_0
# if False:
#     model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# # Remember to go to https://huggingface.co/settings/tokens for a token!
# # And change hf to your username!
# if False:
#     model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# # Save to 16bit GGUF
# if False:
#     model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
# if False: # Pushing to HF Hub
#     model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# # Save to q4_k_m GGUF
# if False:
#     model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
# if False: # Pushing to HF Hub
#     model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# # Save to multiple GGUF options - much faster if you want multiple!
# if False:
#     model.push_to_hub_gguf(
#         "HF_USERNAME/qwen_finetune", # Change hf to your username!
#         tokenizer,
#         quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
#         token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
#     )

# GRPO Reinforcement Learning

In [26]:
# CELL 41 — Add new LoRA on top of SFT model for GRPO
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_alpha = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [27]:
!pip install antlr4-python3-runtime==4.11.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.


In [28]:
# CELL 42 — Load competition data
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import json, re, sys

data = [json.loads(line) for line in open("/content/drive/MyDrive/151B_SP26_Competition-main/data/public.jsonl")]

SYSTEM_PROMPT_MATH = (
    "Solve the math problem. Show only the necessary reasoning. "
    "End with the final answer inside \\boxed{}. "
    "If there are multiple answers, put them comma-separated inside one box, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "Solve the multiple-choice math problem. "
    "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
)

def format_for_grpo(item):
    system = SYSTEM_PROMPT_MCQ if item.get("options") else SYSTEM_PROMPT_MATH
    user = item["question"]
    if item.get("options"):
        opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(item["options"]))
        user = f"{user}\n\nOptions:\n{opts}"

    # Normalize answer to always be a string
    answer = item["answer"]
    if isinstance(answer, list):
        answer = str(answer[0])
    else:
        answer = str(answer)

    return {
        "prompt": [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        "answer": answer,
        "is_mcq": bool(item.get("options"))
    }

grpo_dataset = Dataset.from_list([format_for_grpo(d) for d in data])
print(f"GRPO dataset: {len(grpo_dataset)} examples")

GRPO dataset: 1126 examples


In [29]:
# CELL 43 — Reward function (no judger)
def extract_letter(text):
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def reward_fn(completions, answer, is_mcq, **kwargs):
    rewards = []
    for completion, gold, mcq in zip(completions, answer, is_mcq):
        response = completion[0]["content"] if isinstance(completion, list) else completion
        if mcq:
            correct = extract_letter(response) == str(gold).strip().upper()
        else:
            gold_str = str(gold[0] if isinstance(gold, list) else gold).strip()
            correct = gold_str.lower() in response.lower()
        rewards.append(1.0 if correct else 0.0)
    return rewards

In [30]:
# CELL 44 — GRPO training
trainer = GRPOTrainer(
    model = model,
    tokenizer = tokenizer,
    reward_funcs = reward_fn,
    args = GRPOConfig (
      use_vllm = False,
      learning_rate = 5e-6,
      per_device_train_batch_size = 2,
      gradient_accumulation_steps = 4,
      num_generations = 4,
      max_prompt_length = 256,
      max_completion_length = 1024,
      max_steps = 300,
      save_steps = 100,
      output_dir = "/content/drive/MyDrive/grpo_checkpoints",
      logging_steps = 10,
      report_to = "none",
      seed = 3407,
  ),
    train_dataset = grpo_dataset,
)

trainer.train()

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 2 to the `num_generations` of 6


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,126 | Num Epochs = 4 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 4 x 1) = 24
 "-____-"     Trainable parameters = 16,515,072 of 4,038,983,168 (0.41% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 262144, 'temperature': 0.6, 'top_p': 0.95}. If this is not desired, please set these values explicitly.


KeyboardInterrupt: 

In [ ]:
from google.colab import userdata
token = userdata.get('HT_TOKEN')
model.save_pretrained("/content/drive/MyDrive/grpo_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/grpo_lora")
model.push_to_hub("dcraver2005/grpo_lora", token=token)
tokenizer.push_to_hub("dcraver2005/grpo_lora", token=token)
model.push_to_hub_merged("dcraver2005/grpo_16bit", tokenizer, save_method="merged_16bit", token=token)

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).